In [1]:
%pip install librosa

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install resampy


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install python_speech_features

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
%pip install tensorflow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3.1 -> 23.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [5]:
import os
from tqdm import tqdm
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import wavfile
from python_speech_features import mfcc, logfbank
import resampy

In [6]:
import matplotlib.pyplot as plt
%matplotlib inline

In [7]:
import IPython.display as ipd
import librosa
import librosa.display

In [8]:
!dir

 Volume in drive G is Local Disk
 Volume Serial Number is D6AB-2C6D

 Directory of g:\Downloads\PGDBA\Project audi_classif

15-05-2023  22:37    <DIR>          .
14-05-2023  20:17    <DIR>          ..
20-05-2023  19:01            78,137 audi copy.ipynb
15-05-2023  22:04            72,033 audi.ipynb
15-05-2023  22:01                 0 audi2.ipynb
14-05-2023  20:17    <DIR>          PGDBA-Project-20230514T144324Z-001
14-05-2023  20:15        98,800,322 PGDBA-Project-20230514T144324Z-001.zip
15-05-2023  21:40    <DIR>          saved_models
15-05-2023  14:11    <DIR>          UrbanSound8K
15-05-2023  01:23     6,023,741,708 UrbanSound8K.tar.gz
               5 File(s)  6,122,692,200 bytes
               5 Dir(s)  222,612,131,840 bytes free


In [9]:
import pandas as pd

metadata=pd.read_csv('G:/Downloads/PGDBA/Project audi_classif/UrbanSound8K/UrbanSound8K/metadata/UrbanSound8K.csv')
metadata.head(10)

,slice_file_name,fsID,start,end,salience,fold,classID,class
0,100032-3-0-0.wav,100032,0.000000,0.317551,1,5,3,dog_bark
1,100263-2-0-117.wav,100263,58.500000,62.500000,1,5,2,children_playing
2,100263-2-0-121.wav,100263,60.500000,64.500000,1,5,2,children_playing
3,100263-2-0-126.wav,100263,63.000000,67.000000,1,5,2,children_playing
4,100263-2-0-137.wav,100263,68.500000,72.500000,1,5,2,children_playing
5,100263-2-0-143.wav,100263,71.500000,75.500000,1,5,2,children_playing
6,100263-2-0-161.wav,100263,80.500000,84.500000,1,5,2,children_playing
7,100263-2-0-3.wav,100263,1.500000,5.500000,1,5,2,children_playing
8,100263-2-0-36.wav,100263,18.000000,22.000000,1,5,2,children_playing
9,100648-1-0-0.wav,100648,4.823402,5.471927,2,10,1,car_horn


In [10]:
### Check whether the dataset is imbalanced
metadata['class'].value_counts()

dog_bark            1000
children_playing    1000
air_conditioner     1000
street_music        1000
engine_idling       1000
jackhammer          1000
drilling            1000
siren                929
car_horn             429
gun_shot             374
Name: class, dtype: int64

In [11]:
#### Extracting MFCC's For every audio file
import pandas as pd
import os
import librosa

audio_dataset_path='G:/Downloads/PGDBA/Project audi_classif/UrbanSound8K/UrbanSound8K/audio'

In [12]:
def features_extractor(file):
    audio, sample_rate = librosa.load(file_name, res_type='kaiser_fast') 
    mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
    mfccs_scaled_features = np.mean(mfccs_features.T,axis=0)
    
    return mfccs_scaled_features

In [13]:
import numpy as np
from tqdm import tqdm
### Now we iterate through every audio file and extract features 
### using Mel-Frequency Cepstral Coefficients
extracted_features=[]
for index_num,row in tqdm(metadata.iterrows()):
    file_name = os.path.join(os.path.abspath(audio_dataset_path),'fold'+str(row["fold"])+'/',str(row["slice_file_name"]))
    final_class_labels=row["class"]
    data=features_extractor(file_name)
    extracted_features.append([data,final_class_labels])

3555it [05:05, 13.05it/s]c:\Users\robin\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:256: UserWarning: n_fft=2048 is too large for input signal of length=1323
  warnings.warn(
8326it [11:34, 12.13it/s]c:\Users\robin\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:256: UserWarning: n_fft=2048 is too large for input signal of length=1103
  warnings.warn(
8328it [11:35, 11.97it/s]c:\Users\robin\AppData\Local\Programs\Python\Python311\Lib\site-packages\librosa\core\spectrum.py:256: UserWarning: n_fft=2048 is too large for input signal of length=1523
  warnings.warn(
8732it [12:05, 12.04it/s]


In [14]:
### converting extracted_features to Pandas dataframe
extracted_features_df=pd.DataFrame(extracted_features,columns=['feature','class'])
extracted_features_df.head()

,feature,class
0,"[-217.35526, 70.22338, -130.38527, -53.282898,...",dog_bark
1,"[-424.09818, 109.34077, -52.919525, 60.86475, ...",children_playing
2,"[-458.79114, 121.38419, -46.52066, 52.00812, -...",children_playing
3,"[-413.89984, 101.66373, -35.42945, 53.036354, ...",children_playing
4,"[-446.60352, 113.68541, -52.402214, 60.302044,...",children_playing


In [15]:
### Split the dataset into independent and dependent dataset
X=np.array(extracted_features_df['feature'].tolist())
y=np.array(extracted_features_df['class'].tolist())

In [16]:
X.shape

(8732, 40)

In [17]:
y

array(['dog_bark', 'children_playing', 'children_playing', ...,
       'car_horn', 'car_horn', 'car_horn'], dtype='<U16')

In [18]:
### Label Encoding
y=np.array(pd.get_dummies(y))

In [19]:
y.shape

(8732, 10)

In [20]:
### Train Test Split
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=41)

In [21]:
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(6985, 40) (1747, 40) (6985, 10) (1747, 10)


In [22]:
import tensorflow as tf

In [23]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Dropout,Activation,Flatten
from tensorflow.keras.optimizers import Adam
from sklearn import metrics

In [24]:
### No of classes
num_labels=y.shape[1]
num_labels

10

In [25]:
model=Sequential()
###first layer
model.add(Dense(100,input_shape=(40,)))
model.add(Activation('relu'))
model.add(Dropout(0.5))
###second layer
model.add(Dense(200))
model.add(Activation('relu'))
model.add(Dropout(0.5))
###third layer
model.add(Dense(100))
model.add(Activation('relu'))
model.add(Dropout(0.5))

###final layer
model.add(Dense(num_labels))
model.add(Activation('softmax'))

In [26]:
model.summary()

Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 dense (Dense)               (None, 100)               4100      
                                                                 
 activation (Activation)     (None, 100)               0         
                                                                 
 dropout (Dropout)           (None, 100)               0         
                                                                 
 dense_1 (Dense)             (None, 200)               20200     
                                                                 
 activation_1 (Activation)   (None, 200)               0         
                                                                 
 dropout_1 (Dropout)         (None, 200)               0         
                                                                 
 dense_2 (Dense)             (None, 100)               2

In [27]:
model.compile(loss='categorical_crossentropy',metrics=['accuracy'],optimizer='adam')

In [28]:
## Training the model
from tensorflow.keras.callbacks import ModelCheckpoint
from datetime import datetime 

num_epochs = 100
num_batch_size = 32

checkpointer = ModelCheckpoint(filepath='saved_models/audio_classification.hdf5', 
                               verbose=1, save_best_only=True)
start = datetime.now()

model.fit(X_train, y_train, batch_size=num_batch_size, epochs=num_epochs, validation_data=(X_test, y_test), callbacks=[checkpointer], verbose=1)


duration = datetime.now() - start
print("Training completed in time: ", duration)

Epoch 1/100
210/219 [===========================>..] - ETA: 0s - loss: 11.2849 - accuracy: 0.1179
Epoch 1: val_loss improved from inf to 2.29004, saving model to saved_models\audio_classification.hdf5
219/219 [==============================] - 2s 3ms/step - loss: 10.9692 - accuracy: 0.1181 - val_loss: 2.2900 - val_accuracy: 0.1265
Epoch 2/100
185/219 [========================>.....] - ETA: 0s - loss: 2.5801 - accuracy: 0.1236
Epoch 2: val_loss improved from 2.29004 to 2.27787, saving model to saved_models\audio_classification.hdf5
219/219 [==============================] - 0s 2ms/step - loss: 2.5524 - accuracy: 0.1247 - val_loss: 2.2779 - val_accuracy: 0.1231
Epoch 3/100
215/219 [============================>.] - ETA: 0s - loss: 2.3418 - accuracy: 0.1193
Epoch 3: val_loss improved from 2.27787 to 2.24019, saving model to saved_models\audio_classification.hdf5
219/219 [==============================] - 0s 2ms/step - loss: 2.3402 - accuracy: 0.1188 - val_loss: 2.2402 - val_accuracy: 0.13

In [29]:
test_accuracy=model.evaluate(X_test,y_test,verbose=0)
print(test_accuracy[1])

0.7240984439849854


In [30]:
predict_x=model.predict(X_test) 
classes_x=np.argmax(predict_x,axis=1)

55/55 [==============================] - 0s 578us/step


In [31]:
Y_preds = model.predict

In [32]:
filename="G:/Downloads/PGDBA/Project audi_classif/UrbanSound8K/UrbanSound8K/audio/fold2/4201-3-0-0.wav"
audio, sample_rate = librosa.load(filename, res_type='kaiser_fast') 
mfccs_features = librosa.feature.mfcc(y=audio, sr=sample_rate, n_mfcc=40)
mfccs_scaled_features = np.mean(mfccs_features.T,axis=0)

print(mfccs_scaled_features)
mfccs_scaled_features=mfccs_scaled_features.reshape(1,-1)
print(mfccs_scaled_features)
print(mfccs_scaled_features.shape)
predictions = (model.predict(X_test) > 0.5).astype("int32")
print(predictions)


[-2.33868500e+02  1.09056206e+02 -5.76975298e+00  1.65992870e+01
 -3.46133270e+01  2.37382679e+01 -4.64467859e+00  2.05459900e+01
  5.89833069e+00  9.75115967e+00  2.91429877e+00  1.32354250e+01
 -5.70944548e-02  1.65461006e+01 -5.76831675e+00  1.81195545e+01
  8.35268211e+00  2.35184250e+01  5.55858326e+00  5.03801680e+00
 -3.14165020e+00  8.64437008e+00 -1.87388659e+00  3.97579122e+00
  7.93383360e+00  1.31894326e+00 -7.96618462e+00  8.11050510e+00
  3.68082595e+00  1.63433247e+01  5.00579178e-01  7.89288104e-01
  1.75276268e+00 -3.20594263e+00  4.10227823e+00  2.05818057e+00
 -2.51707935e+00  3.52652121e+00  1.53722748e-01  1.73329616e+00]
[[-2.33868500e+02  1.09056206e+02 -5.76975298e+00  1.65992870e+01
  -3.46133270e+01  2.37382679e+01 -4.64467859e+00  2.05459900e+01
   5.89833069e+00  9.75115967e+00  2.91429877e+00  1.32354250e+01
  -5.70944548e-02  1.65461006e+01 -5.76831675e+00  1.81195545e+01
   8.35268211e+00  2.35184250e+01  5.55858326e+00  5.03801680e+00
  -3.14165020e+00  